# Pipeline de Extracción de Datos Comunes a cada Capital/País

Este *notebook* implementa un flujo completo de extracción, transformación y carga (ETL) para obtener un primer dataset que contenga características propias de cada país/capital europea. Estas características son:

1. Latitud
2. Longitud
3. Altitud
4. Superficie
5. Distancia al mar
6. Factor cuenca

Estas variables se consiguen a partir de diversas técnicas de extracción: llamadas a APIs, realizando *Web Scraping* y a partir de cálculos matemáticos.

**Llamadas a APIs**

Una API (*Application Programming Interface*) es un conjunto de reglas que permite que dos aplicaciones se comuniquen entre sí. En este proyecto, nosotros actuamos como un "cliente" que solicita información específica a un "servidor" externo. En este script, enviamos una petición estructurada (normalmente una URL con parámetros como la latitud y longitud) y el servidor responde con los datos exactos en un formato ligero y fácil de leer para la máquina, generalmente JSON.

***Web Scraping***

El *Web Scraping* es una técnica que se utiliza cuando los datos que necesitamos están publicados en una página web pero no existe una API oficial para descargarlos de forma estructurada. En este script, leemos el código HTML de una página web (en este caso, Wikipedia) como si fuera un navegador, pero en lugar de mostrar imágenes y texto, buscamos etiquetas específicas donde está escondida la información. Utilizamos *BeautifulSoup* para navegar por el árbol de etiquetas HTML y *Requests* para descargar el contenido de la web.

In [ ]:
# =============================================================================
# LIBRERÍAS DE CIENCIA DE DATOS Y COMPUTACIÓN NUMÉRICA
# =============================================================================
import numpy as np           # Realiza cálculos matriciales y operaciones matemáticas vectorizadas.
import math                  # Proporciona funciones matemáticas fundamentales (seno, coseno, raíz cuadrada).
import pandas as pd          # Estructura los datos en DataFrames; esencial para limpieza y unión (merge) de archivos.

# =============================================================================
# LIBRERÍAS DE COMUNICACIÓN Y EXTRACCIÓN DE DATOS (SCRAPING)
# =============================================================================
import requests              # Gestiona las peticiones HTTP para descargar datos de Wikipedia y APIs externas.
from bs4 import BeautifulSoup # Analiza el código HTML de las webs para extraer datos específicos (como coordenadas).
import re                    # Expresiones Regulares: permite limpiar y extraer números de cadenas de texto complejas.

# =============================================================================
# LIBRERÍAS DE GESTIÓN DEL SISTEMA Y FLUJO DE TRABAJO
# =============================================================================
import os                    # Administra rutas de archivos y carpetas de forma compatible entre Windows/Linux.
import time                  # Controla los tiempos de ejecución y pausas (sleep) para evitar bloqueos de APIs.

# =============================================================================
# LIBRERÍAS DE ANÁLISIS GEOESPACIAL Y CARTOGRAFÍA (EL CORE TÉCNICO)
# =============================================================================

# Cartopy permite leer y manejar archivos de mapas profesionales (Shapefiles) de Natural Earth.
import cartopy.io.shapereader as shpreader 

# Shapely se encarga de la lógica geométrica:
# - Point: Representa la ciudad como un punto.
# - MultiPolygon: Maneja formas complejas como los océanos.
# - nearest_points: Encuentra la distancia mínima entre dos geometrías.
from shapely.geometry import Point, MultiPolygon
from shapely.ops import nearest_points

# Geopy calcula la distancia geodésica real (en km) sobre la curvatura de la Tierra (elipsoide).
from geopy.distance import geodesic

In [ ]:
# =============================================================================
# CONFIGURACIÓN E INICIALIZACIÓN DEL ENTORNO
# =============================================================================

# Definimos la ruta de salida.
RUTA_GUARDADO = r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivo_paises"

# Este es un diccionario de los países y sus identificadores. 
# Usamos el código ISO como identificador único para futuros cruces de datos.
# El valor (nombre) es la clave de búsqueda exacta para generar la URL de Wikipedia.
locations = {
    "AD": "Andorra", "AL": "Albania", "AT": "Viena", "BA": "Bosnia y Herzegovina",
    "BE": "Bruselas", "BG": "Sofía", "CH": "Berna", "CY": "Nicosia",
    "CZ": "Praga", "DE": "Berlín", "DK": "Copenhague", "EE": "Tallín",
    "ES": "Madrid", "FI": "Helsinki", "FR": "París", "GR": "Atenas",
    "HR": "Zagreb", "HU": "Budapest", "IE": "Dublín", "IS": "Reikiavik",
    "IT": "Roma", "LT": "Vilna", "LU": "Luxemburgo", "LV": "Riga",
    "ME": "Montenegro", "MK": "Macedonia del Norte", "MT": "La Valeta",
    "NL": "Ámsterdam", "NO": "Oslo", "PL": "Varsovia", "PT": "Lisboa",
    "RO": "Bucarest", "RS": "Serbia", "SE": "Estocolmo", "SI": "Liubliana",
    "SK": "Bratislava", "XK": "Kosovo"
}

In [ ]:
# =============================================================================
# FUNCIONES DE EXTRACCIÓN (WEB SCRAPING Y CONEXIÓN CON APIS)
# =============================================================================

def get_wiki_coordinates(nombre):
    """
    EXTRACCIÓN DE COORDENADAS MEDIANTE SCRAPING:
    Wikipedia almacena las coordenadas en varios formatos. El formato decimal es el más preciso
    para cálculos matemáticos posteriores.
    """
    # Construimos la URL dinámica sustituyendo espacios por guiones bajos
    url = f"https://es.wikipedia.org/wiki/{nombre.replace(' ', '_')}"
    
    # El 'User-Agent' es crucial: simula que somos un navegador real (Chrome/Firefox).
    # Sin esto, los servidores de Wikipedia podrían bloquear la petición al detectarla como un 'bot'.
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    try:
        res = requests.get(url, headers=headers, timeout=5)
        # Parseamos el HTML con BeautifulSoup para poder navegar por el árbol de etiquetas
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # Buscamos la etiqueta específica <span class="geo-dec"> que contiene la Lat/Lon decimal.
        # Este dato es inyectado por Wikipedia mediante plantillas geográficas.
        geo_span = soup.find('span', {'class': 'geo-dec'})
        
        if geo_span:
            texto_coords = geo_span.get_text()
            # RE (Regular Expressions): Buscamos cualquier cadena que parezca un número decimal.
            # El patrón busca opcionalmente un signo (+ o -), dígitos, un punto y más dígitos.
            nums = re.findall(r"[-+]?\d*\.\d+|\d+", texto_coords)
            
            if len(nums) >= 2:
                lat, lon = float(nums[0]), float(nums[1])
                
                # LÓGICA DE HEMISFERIOS:
                # En coordenadas decimales, el Oeste (W/O) y el Sur (S) son negativos.
                if 'O' in texto_coords or 'W' in texto_coords: lon = -abs(lon)
                if 'S' in texto_coords: lat = -abs(lat)
                return lat, lon
    except Exception as e:
        print(f"Error extrayendo coordenadas para {nombre}: {e}")
    return None, None

In [ ]:
def get_altitude(lat, lon):
    """
    CONEXIÓN CON API EXTERNA (OPEN-METEO):
    La altitud no siempre es fiable en Wikipedia (a veces es media, a veces máxima).
    Usamos una API profesional que cruza la Lat/Lon con modelos de elevación digital (DEM).
    """
    if lat is None or lon is None: return None
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    try:
        r = requests.get(url, timeout=5)
        # La respuesta es un JSON. Extraemos el valor del primer índice de la lista 'elevation'.
        return r.json()['elevation'][0] if r.status_code == 200 else None
    except: return None

In [ ]:
def get_wiki_surface(nombre):
    """
    EXTRACCIÓN DE SUPERFICIE (INFOBOX SCRAPING):
    Este dato es fundamental para calcular el radio de influencia de cada lugar.
    Buscamos dentro de la tabla lateral (infobox) de la Wikipedia.
    """
    url = f"https://es.wikipedia.org/wiki/{nombre.replace(' ', '_')}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        res = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(res.text, 'html.parser')
        infobox = soup.find('table', {'class': 'infobox'})
        if not infobox: return None
        
        # Iteramos por las filas de la tabla buscando las palabras clave
        for fila in infobox.find_all('tr'):
            texto_fila = fila.get_text().lower()
            if 'superficie' in texto_fila or 'área' in texto_fila:
                # El valor suele estar en la celda de datos (td). 
                # Si la fila solo tiene el título, buscamos en la siguiente fila.
                td = fila.find('td') or fila.find_next_sibling('tr').find('td')
                if td:
                    raw_val = td.get_text().strip()
                    # Limpiamos el texto: quitamos espacios raros (\xa0) y convertimos 
                    # formatos europeos (1.234,56) a formato Python (1234.56).
                    match = re.search(r'([\d\s\.]+,?\d*)', raw_val)
                    if match:
                        num_str = match.group(1).replace('\xa0', '').replace(' ', '')
                        if '.' in num_str and ',' in num_str:
                            num_str = num_str.replace('.', '').replace(',', '.')
                        elif ',' in num_str:
                            num_str = num_str.replace(',', '.')
                        return float(num_str)
        return None
    except: return None

In [ ]:
# =============================================================================
# ANÁLISIS GEOESPACIAL (CÁLCULOS DERIVADOS)
# =============================================================================

# CARGA DE DATOS GEOGRÁFICOS FÍSICOS:
# Descargamos el shapefile de océanos de Natural Earth. 
# Esto nos permite tener una "máscara" del agua en todo el mundo.
ocean_shp = shpreader.natural_earth(resolution='110m', category='physical', name='ocean')
ocean_geom = MultiPolygon(list(shpreader.Reader(ocean_shp).geometries()))

def get_distancia_mar(lat, lon):
    """
    CÁLCULO DE PROXIMIDAD COSTERA:
    La cercanía al mar influye en la dispersión de contaminantes (brisas marinas).
    """
    if lat is None or lon is None: return None
    punto_ciudad = Point(lon, lat) # Creamos un objeto punto (X=Longitud, Y=Latitud)
    
    # Verificamos si el punto ya cae dentro del polígono del océano
    if ocean_geom.contains(punto_ciudad): return 0.0
    
    # Buscamos el punto más cercano en la costa (p2) respecto a nuestra ciudad (punto_ciudad)
    p1, p2 = nearest_points(punto_ciudad, ocean_geom)
    
    # La librería Geopy calcula la distancia real en KM sobre una esfera (no plano).
    return round(geodesic((lat, lon), (p2.y, p2.x)).km, 2)

In [ ]:
def get_factor_cuenca(lat, lon, alt_centro, superficie_km2):
    """
    ÍNDICE DE POSICIÓN TOPOGRÁFICA (TPI):
    Este cálculo determina si una ciudad está en un "agujero" (valle) o en una zona despejada.
    Las ciudades en cuencas (TPI negativo) suelen atrapar más la contaminación.
    """
    if lat is None or lon is None or alt_centro is None: return 0
    
    # ESTIMACIÓN DEL RADIO DE MUESTREO:
    # Si tenemos la superficie, calculamos el radio del círculo equivalente (A = π * r²)
    # Multiplicamos por 1.5 para analizar también las montañas que rodean la zona.
    radio_km = math.sqrt(superficie_km2 / math.pi) * 1.5 if superficie_km2 else 10
    
    # Convertimos KM a grados de arco terrestre (1 grado ≈ 111 km).
    radio_grados = radio_km / 111.0
    
    elevaciones_entorno = []
    # MUESTREO CIRCULAR:
    # Generamos 50 puntos alrededor de la ciudad usando trigonometría (Seno y Coseno).
    # Esto crea un anillo de datos de elevación.
    for i in range(50):
        angulo = 2 * math.pi * i / 50
        # p_lat y p_lon definen un punto en el perímetro del círculo de búsqueda
        p_lat = lat + radio_grados * math.cos(angulo)
        p_lon = lon + radio_grados * math.sin(angulo)
        
        alt = get_altitude(p_lat, p_lon)
        if alt is not None: elevaciones_entorno.append(alt)
    
    if not elevaciones_entorno: return 0
    
    # LÓGICA DEL TPI:
    # TPI = Altitud de la ciudad - Media de las altitudes de alrededor.
    # Si el resultado es -100, significa que la ciudad está 100 metros por debajo de su entorno.
    promedio_entorno = sum(elevaciones_entorno) / len(elevaciones_entorno)
    return round(alt_centro - promedio_entorno, 2)

In [ ]:
# =============================================================================
# PROCESAMIENTO POR LOTES (MAIN LOOP)
# =============================================================================

lista_resultados = []

for iso, nombre in locations.items():
    print(f"🔎 Analizando {nombre} ({iso})...")
    
    # PASO A: El scraping de coordenadas es la llave maestra. Sin esto no hay mapa.
    lat, lon = get_wiki_coordinates(nombre)
    
    if lat is not None:
        # PASO B: Con las coordenadas obtenidas, disparamos el resto de peticiones.
        alt = get_altitude(lat, lon)
        sup = get_wiki_surface(nombre)
        mar = get_distancia_mar(lat, lon)
        tpi = get_factor_cuenca(lat, lon, alt, sup)
        
        # Estructuramos el registro para el DataFrame
        lista_resultados.append({
            "Pais_ISO": iso,
            "Pais": nombre,
            "Latitud": lat,
            "Longitud": lon,
            "Altitud (m)": alt,
            "Superficie (km^2)": sup,
            "Distancia_mar (km)": mar,
            "TPI_Cuenca": tpi
        })
    else:
        print(f" ❌ Error: Wikipedia no devolvió coordenadas válidas para {nombre}")
    
    # CONTROL DE FLUJO:
    # Pausamos medio segundo entre peticiones para evitar que nos baneen por tráfico abusivo.
    time.sleep(0.5)

In [ ]:
# =============================================================================
# EXPORTACIÓN Y CIERRE
# =============================================================================

# Creamos el objeto DataFrame, que es como una tabla de Excel en memoria de Python.
df = pd.DataFrame(lista_resultados)

# Guardamos el resultado en CSV.
# Usamos 'utf-8-sig' porque Excel en español a veces no lee bien los acentos si se guarda en utf-8 normal.
nombre_archivo = "países.csv"
df.to_csv(os.path.join(RUTA_GUARDADO, nombre_archivo), index=False, encoding='utf-8-sig')

print(f"\n🏆 ¡PROCESO COMPLETADO!")
print(f"Se ha generado el archivo '{nombre_archivo}'.")


🏆 ¡PROCESO COMPLETADO!
Se ha generado el archivo 'países.csv'.


### Superficie + altitud + distancia al mar

In [1]:
import numpy as np
import math
import pandas as pd
import requests
import time
import re
import os
from bs4 import BeautifulSoup
import cartopy.io.shapereader as shpreader
from shapely.geometry import Point, MultiPolygon
from shapely.ops import nearest_points
from geopy.distance import geodesic

# 1. Configuración y Datos iniciales
RUTA_GUARDADO = r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM"
locations = {
    "AD": (42.546245, 1.601554, "Andorra"), "AL": (41.153332, 20.168331, "Albania"),
    "AT": (48.20849, 16.37208, "Viena"), "BA": (43.915886, 17.679076, "Bosnia y Herzegovina"),
    "BE": (50.85045, 4.34878, "Bruselas"), "BG": (42.69751, 23.32415, "Sofía"),
    "CH": (46.94809, 7.44744, "Berna"), "CY": (35.17284, 33.35397, "Nicosia"),
    "CZ": (50.08804, 14.42076, "Praga"), "DE": (52.52437, 13.41053, "Berlín"),
    "DK": (55.67594, 12.56553, "Copenhague"), "EE": (59.43696, 24.75353, "Tallín"),
    "ES": (40.4165, -3.70256, "Madrid"), "FI": (60.16952, 24.93545, "Helsinki"),
    "FR": (48.85341, 2.3488, "París"), "GR": (37.98376, 23.72784, "Atenas"),
    "HR": (45.81444, 15.97798, "Zagreb"), "HU": (47.49835, 19.04045, "Budapest"),
    "IE": (53.33306, -6.24889, "Dublín"), "IS": (64.13548, -21.89541, "Reikiavik"),
    "IT": (41.89193, 12.51133, "Roma"), "LT": (54.68916, 25.2798, "Vilna"),
    "LU": (49.815273, 6.129583, "Luxemburgo"), "LV": (56.946, 24.10589, "Riga"),
    "ME": (42.708678, 19.37439, "Montenegro"), "MK": (41.608635, 21.745275, "Macedonia del Norte"),
    "MT": (35.89968, 14.5148, "La Valeta"), "NL": (52.37403, 4.88969, "Ámsterdam"),
    "NO": (59.91273, 10.74609, "Oslo"), "PL": (52.22977, 21.01178, "Varsovia"),
    "PT": (38.72509, -9.1498, "Lisboa"), "RO": (44.43225, 26.10626, "Bucarest"),
    "RS": (44.016521, 21.005859, "Serbia"), "SE": (59.32938, 18.06871, "Estocolmo"),
    "SI": (46.05108, 14.50513, "Liubliana"), "SK": (48.14816, 17.10674, "Bratislava"),
    "XK": (42.602636, 20.902977, "Kosovo")
}

# --- FUNCIONES DE APOYO ---

def get_altitude(lat, lon):
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    try:
        r = requests.get(url, timeout=5)
        return r.json()['elevation'][0] if r.status_code == 200 else None
    except: return None

def get_wiki_surface(nombre):
    url = f"https://es.wikipedia.org/wiki/{nombre.replace(' ', '_')}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        res = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(res.text, 'html.parser')
        infobox = soup.find('table', {'class': 'infobox'})
        if not infobox: return None
        filas = infobox.find_all('tr')
        for i, fila in enumerate(filas):
            texto_fila = fila.get_text().lower()
            if 'superficie' in texto_fila or 'área' in texto_fila:
                target_fila = fila
                if not any(c.isdigit() for c in fila.get_text()):
                    if i + 1 < len(filas): target_fila = filas[i+1]
                td = target_fila.find('td')
                if td:
                    raw_val = td.get_text().strip()
                    match = re.search(r'([\d\s\.]+,?\d*)', raw_val)
                    if match:
                        num_str = match.group(1).replace('\xa0', '').replace(' ', '')
                        if '.' in num_str and ',' in num_str:
                            num_str = num_str.replace('.', '').replace(',', '.')
                        elif ',' in num_str:
                            num_str = num_str.replace(',', '.')
                        return float(num_str)
        return None
    except: return None

# Carga de Océanos para Distancia al Mar
ocean_shp = shpreader.natural_earth(resolution='110m', category='physical', name='ocean')
ocean_geom = MultiPolygon(list(shpreader.Reader(ocean_shp).geometries()))

def get_distancia_mar(lat, lon):
    punto_ciudad = Point(lon, lat)
    if ocean_geom.contains(punto_ciudad): return 0.0
    p1, p2 = nearest_points(punto_ciudad, ocean_geom)
    return round(geodesic((lat, lon), (p2.y, p2.x)).km, 2)

def get_factor_cuenca(lat, lon, alt_centro, superficie_km2):
    if alt_centro is None or superficie_km2 is None:
        # Si no hay superficie, usamos un radio estándar de 10km
        radio_busqueda_km = 10
    else:
        # Calculamos el radio de la ciudad (r = sqrt(Area/pi)) 
        # y le sumamos un 50% extra para detectar el relieve circundante
        radio_ciudad = math.sqrt(superficie_km2 / math.pi)
        radio_busqueda_km = radio_ciudad * 1.5

    # Convertimos km a grados aprox (1 grado = 111km)
    radio_grados = radio_busqueda_km / 111.0
    
    elevaciones_entorno = []
    num_puntos = 100

    
    for i in range(num_puntos):
        # Ángulo en radianes para los 100 puntos
        angulo = 2 * math.pi * i / num_puntos
        
        # Calculamos la coordenada del punto en el borde
        p_lat = lat + radio_grados * math.cos(angulo)
        p_lon = lon + radio_grados * math.sin(angulo)
        
        # Pedimos la altitud a la API
        alt = get_altitude(p_lat, p_lon)
        if alt is not None:
            elevaciones_entorno.append(alt)
        
        # Pequeña pausa cada 20 puntos para no saturar la API gratuita
        if i % 20 == 0:
            time.sleep(0.2)
            
    if not elevaciones_entorno:
        return 0
    
    promedio_entorno = sum(elevaciones_entorno) / len(elevaciones_entorno)
    tpi = alt_centro - promedio_entorno
    
    return round(tpi, 2)

# --- 3. EJECUCIÓN FINAL ---

lista_paises = []
print("🚀 Iniciando extracción final (Altitud, Superficie, Mar y Cuenca)...")

for iso, datos in locations.items():
    lat, lon, nombre = datos
    alt = get_altitude(lat, lon)
    sup = get_wiki_surface(nombre)
    mar = get_distancia_mar(lat, lon)
    cuenca = get_factor_cuenca(lat, lon, alt, sup)
    
    lista_paises.append({
        "Pais_ISO": iso,
        "Capital/Pais": nombre,
        "Latitud": lat,
        "Longitud": lon,
        "Altitud (m)": alt,
        "Superficie (km2)": sup,
        "Distancia al mar (km)": mar,
        "Factor Cuenca (TPI)": cuenca
    })
    print(f"✔️ {nombre}: Alt {alt}m | Sup {sup} km^2 | Mar {mar}km | TPI {cuenca}")
    time.sleep(0.2)

# Guardado
df = pd.DataFrame(lista_paises)
df.to_csv(os.path.join(RUTA_GUARDADO, "países.csv"), index=False, encoding='utf-8-sig')

print("\n✅ ¡Dataset completado!")

🚀 Iniciando extracción final (Altitud, Superficie, Mar y Cuenca)...
✔️ Andorra: Alt 1943.0m | Sup 178.0 km^2 | Mar 139.75km | TPI -244.05
✔️ Albania: Alt 149.0m | Sup 144.0 km^2 | Mar 66.67km | TPI -501.84
✔️ Viena: Alt 196.0m | Sup 414.78 km^2 | Mar 365.93km | TPI -33.79
✔️ Bosnia y Herzegovina: Alt 1750.0m | Sup 128.0 km^2 | Mar 103.03km | TPI 487.02
✔️ Bruselas: Alt 27.0m | Sup 32.61 km^2 | Mar 93.01km | TPI -29.76
✔️ Sofía: Alt 555.0m | Sup 492.0 km^2 | Mar 225.22km | TPI -325.6
✔️ Berna: Alt 554.0m | Sup 51.62 km^2 | Mar 311.5km | TPI -74.55
✔️ Nicosia: Alt 154.0m | Sup 51.06 km^2 | Mar 22.88km | TPI -2.87
✔️ Praga: Alt 205.0m | Sup 496.0 km^2 | Mar 408.76km | TPI -79.25
✔️ Berlín: Alt 46.0m | Sup 891.69 km^2 | Mar 145.16km | TPI -8.3
✔️ Copenhague: Alt 11.0m | Sup 77.2 km^2 | Mar 5.56km | TPI 3.32
✔️ Tallín: Alt 10.0m | Sup 159.2 km^2 | Mar 5.08km | TPI -14.16
✔️ Madrid: Alt 651.0m | Sup 604.45 km^2 | Mar 335.33km | TPI 5.5
✔️ Helsinki: Alt 12.0m | Sup 715.48 km^2 | Mar 2.23km | 